# River regulation
***

***Author:** Chus Casado Rodríguez*<br>
***Date:** 24-09-2026*<br>

**Introduction:**<br>

In this notebook I create raster of reservoir regulation that can be used to discern the stations/reservoirs in CAMELS-ES/BEAVERS-ES whose inflow is in (semi)natural regime.

I also create a graph of the river network using only the reservoirs in BEAVERS-ES to identify headwater reservoirs..

**To be improved**


In [1]:
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
from rasterio.features import rasterize

import pyflwdir
import networkx as nx

from utils import merge_points, np2xr
from ocab.config import Config
from ocab.graph import *

## Config

In [2]:
cfg_beavers = Config('../BEAVERS/config_BEAVERS_v100.yml')

meteo = 'ROCIO-IBEB'
path_meteo = cfg_beavers.path_meteo.parent.parent / meteo / 'GIS'

## Data

### Digital Elevation Model

In [3]:
# load flow direction map and convert it to `flwdir`
flwdir_da = rxr.open_rasterio(cfg_beavers.path_merit / 'dir.tif').squeeze(dim='band')
crs = flwdir_da.rio.crs
flwdir = pyflwdir.from_array(
    flwdir_da.data,
    ftype='d8',
    transform=flwdir_da.rio.transform(),
    check_ftype=False,
    latlon=True
)

# compute pixel area in km²
pixarea = flwdir.area * 1e-6

# Compute upstream area map in km²
uparea = flwdir.upstream_area(unit='km2')
mask_area = uparea >= cfg_beavers.area_min

# Map of the Strahler order
strahler = flwdir.stream_order(type='strahler', mask=mask_area)

### Points

#### Dams

In [4]:
# import dams
dams = load_MERIT_points(
    path=cfg_beavers.path_dataset / 'preprocessing' / 'basins' / 'output' / 'dams_outlets_3sec.geojson',
    flwdir=flwdir,
    crs=crs,
    kind='dam'
)
print(f'{len(dams)} dams')

# import reservoirs
reservoirs = gpd.read_file(
    '../../docs/layers/reservoirs.geojson',
    columns=['id', 'cap_mcm']
)
reservoirs['kind'] = 'dam'
reservoirs.set_index(['id', 'kind'], inplace=True)
print(f'{len(reservoirs)} reservoirs')

# add reservoir capacity to the dams
dams = pd.concat([dams, reservoirs['cap_mcm']], axis=1)

366 dams
366 reservoirs


### Meteorology

In [5]:
precip_file = cfg_beavers.path_gis / "areal_precipitation.tif"
if precip_file.is_file():
    # load file
    precip_areal = rxr.open_rasterio(precip_file).squeeze('band').drop_vars('band')
else:
    # load ROCIO-IBEB mean daily precipitation (mm/d)
    precip = rxr.open_rasterio(
        path_meteo / 'avg_precipitation_wgs84.tif'
        ).squeeze('band').drop_vars('band')

    # reproject to the MERIT grid
    precip_merit = precip.rio.reproject_match(
        flwdir_da,
        resampling='bilinear'
    )
    precip_merit = np.nan_to_num(precip_merit, nan=0.0)

    # accumulated precipitation volume (km³)
    precip_total = flwdir.accuflux(precip_merit * 1e-6 * pixarea)

    # DataArray of areal precipitation over river pixels (mm/d)
    precip_areal = np2xr(
        data=precip_total / uparea * 1e6,
        template=flwdir_da,
        mask=mask_area,
        name='precipitation',
        attrs={'long_name': 'areal precipitation', 'units': 'mm/d'}
    )

    # Set spatial reference and export to GeoTIFF
    precip_areal.rio.to_raster(
        precip_file, 
        dtype="float32", 
        compress="deflate"
    )

## Regulation

### Upstream storage

In [6]:
# raster of reservoir capacity
capacity = rasterize(
    shapes=[(geom, cap) for geom, cap in zip(dams.geometry, dams['cap_mcm'])],
    out_shape=flwdir.shape,
    transform=flwdir.transform,
    fill=0.0,
    dtype=np.float32
)

# accumulated capacity
upcapacity = flwdir.accuflux(capacity)

# convert to DataArray
upcapacity = np2xr(
    data=upcapacity,
    template=flwdir_da,
    mask=mask_area,
    name='capacity',
    attrs=dict(long_name='upstream capacity', units='hm³')
)

# export
upcapacity.rio.to_raster(
    cfg_beavers.path_gis / "upstream_storage.tif", 
    dtype="float32", 
    compress="deflate"
)

### Degree of disruptivity
#### In meters

In [9]:
# compute degree of disruptivity (m)
disruptivity_m = upcapacity / uparea
disruptivity_m = disruptivity_m.rename('disruptivity')
disruptivity_m.attrs.update({'long_name': 'degree of disruptivity', 'units': 'm'})

# export
disruptivity_m.rio.to_raster(
    cfg_beavers.path_gis / "disruptivity_meters.tif", 
    dtype="float32", 
    compress="deflate"
)

#### In days
I normalize the disruptivity by the average upstream daily precipitation.


In [10]:
# normalize disruptivity by avg. daily precipitation (d)
disruptivity_d = disruptivity_m * 1000 / precip_areal
disruptivity_d.attrs.update({'long_name': 'degree of disruptivity', 'units': 'd'})

# export
disruptivity_d.rio.to_raster(
    cfg_beavers.path_gis / "disruptivity_days.tif", 
    dtype="float32", 
    compress="deflate"
)

## Topological Processing

### Outlets

In this section I find the river mouth for every basin in the dataset.

In [ ]:
# find basin outlets
outlets = find_outlets(dams, flwdir, uparea)
print(f'{len(outlets)} outlets')

> **Note**. It would be great to name each outlet after the basin codes: Duero 2000, Tajo 3000, Guadiana 4000, Guadalquivir 5000. The problem is that the IDs are not unique between stations and dams, that in some cases the x000 ID is already taken, and that small catchments in Cantábrico, Galicia Costa or Júcar are difficult to name.

### Find Downstream Point

In [ ]:
# merge all available points
points = merge_points([dams, outlets], keep='dam')
points.loc[points['cap_mcm'].isnull(), 'cap_mcm'] = 0

# add or rename attributes
points['lat'] = points.geometry.y
points['lon'] = points.geometry.x
points.rename(
    columns={
        # 'catch_skm': 'area', 
        'flwdir_index': 'pixel'
    }, 
    inplace=True, 
    errors='ignore'
)
points['strahler'] = strahler.ravel()[points.pixel]

# find points downstream
pixel_to_id = dict(zip(points['pixel'], points.index))
ds_neighbours = [find_downstream_neighbour(pixel, flwdir, pixel_to_id) for pixel in points['pixel']]
neighbours, distances = zip(*ds_neighbours)
points['downstream_ID'] = neighbours
points['dist_to_outlet'] = distances

# compute distance to downstream station
dist_m = pd.Series(index=points.index, name='dist_m')
for ID, ds_ID in tqdm(points['downstream_ID'].items(), total=len(points)):
    # ds_ID = points.loc[ID, 'downstream_ID']
    try:
        if pd.isnull(ds_ID):
            dist_m[ID] = points.loc[ID, 'dist_to_outlet']
        else:
            dist_m[ID] = np.diff(points.loc[[ds_ID, ID], 'dist_to_outlet'])[0]
    except Exception as e:
        print(ID, ds_ID, e)
        break
points['dist_m'] = dist_m

# # export
# # points.to_file(cfg_beavers.path_gis / 'nodes.geojson', driver='GeoJSON')
# points.to_file('nodes.geojson', driver='GeoJSON')

### Subbasin Delineation

Subbasins are the intercatchments between gauging stations and dams, i.e., not the complete upstream catchment, but the subcatchment up to the upstream point (either station or dam).

In [ ]:
subbasins = delineate_subbasins(
    points.query("kind in ['station', 'dam']"), 
    flwdir, 
    uparea
)
print(f'{len(subbasins)} subbasins')

# # export
# subbasins.to_file(cfg_beavers.path_gis / 'subbasins.geojson', driver='GeoJSON')

### Basin Delineation

Basins are the whole river basin polygons. To get those with the same function (`delineate_subbasins`), we use only the `outlets`.

In [ ]:
basins = delineate_subbasins(
    points.xs('outlet', level='kind', drop_level=False), 
    flwdir, 
    uparea, 
    name='basin'
)
print(f'{len(basins)} basins')

# # export
# basins.to_file(cfg_beavers.path_gis / 'basins.geojson', driver='GeoJSON')

### Plot

In [ ]:
# plot stations, dams and subbasins
size = 4
points_kwgs = {
    # 'station': dict(markersize=size),
    'dam': dict(markersize=size, marker='v', c='k'),
    'outlet': dict(markersize=size),
}

fig, ax = plt.subplots()
for kind, kwgs in points_kwgs.items():
    points.xs(kind, level='kind').plot(ax=ax, label=kind, zorder=2, **kwgs)
subbasins.boundary.plot(ax=ax, edgecolor='dimgrey', lw=.2, zorder=0)
basins.boundary.plot(ax=ax, edgecolor='k', lw=.5, zorder=1)
ax.set_aspect('equal')
ax.legend(loc=4, frameon=False)
ax.axis('off');

### Rivers

In [ ]:
# Delineate streams
streams = gpd.GeoDataFrame.from_features(
    flwdir.streams(mask=uparea >= 1000),
    crs=crs
)

# keep only those within the basins
streams = gpd.sjoin(streams, basins, how='inner', predicate='intersects')

# # # export
# # streams.to_file(cfg_beavers.path_gis / 'streams.geojson', driver='GeoJSON')

### Confluences

In [ ]:
# for ID in outlets.index:
#     print(ID)
#     parents = points[points.downstream_ID == ID]
#     if len(parents) < 2:
#         continue
#     else:
#         break
# paths, dists = flwdir.path(parents['pixel'])
# n_parents = len(parents)

# confluences = []
# for i in range(n_parents):
#     for j in range(i + 1, n_parents):
#         print(i, '-', j)
#         for pixel in paths[i]:
#             if pixel in paths[j]:
#                 confluences.append(pixel.item())
#                 break
# confluences = list(set(confluences))

# for ID2 in parents.index:
#     parents2 = points[points.downstream_ID == ID2]
#     if len(parents2) < 2:
#         continue
#     else:
#         print(ID2)
#         break

## Graph

### Create the graph

In [ ]:
# Initialize a Directed Graph
G = nx.DiGraph()

# Add nodes
node_attrs = points[['lat', 'lon', 'catch_skm', 'cap_mcm', 'pixel', 'strahler']].to_dict('index')
for node_id, attrs in node_attrs.items():
    G.add_node(node_id, **attrs)
    
# Add edges
edges = points.dropna(subset=['downstream_ID'])
for ID, row in edges.iterrows():
    G.add_edge(u_of_edge=ID, v_of_edge=row['downstream_ID'], dist_m=row['dist_m'])

print(f"Graph created with {G.number_of_nodes()} stations and {G.number_of_edges()} connections.")

### Explore the graph

In [ ]:
# Find all nodes upstream of a target node
target_node = (2029, 'dam')

upstream_stations = list(nx.ancestors(G, target_node))
print('ancestors:', upstream_stations)
source_node = upstream_stations[-1]

if G.has_node(source_node) and G.has_node(target_node):
    if nx.has_path(G, source_node, target_node):
        dist = nx.shortest_path_length(G, source_node, target_node, weight='dist_m')
        print(f"Distance from {source_node} to {target_node}: {dist/1000:.2f} km")
    else:
        print("Nodes exist, but they are in different river basins!")
else:
    print("One of the station IDs is missing from the graph.")

In [ ]:
# identify outlet nodes
outlets = [node for node, degree in G.out_degree() if degree == 0]
print(f'{len(outlets)} outlet points.')

# identify headwater nodes
headwaters = [node for node, degree in G.in_degree() if degree == 0]
print(f'{len(headwaters)} headwater points.')

In [ ]:
# export TXT file of headwater reservoirs
headwater_ids = [f'{cfg_beavers.prefix}_{ID}' for ID, kind in headwaters]
txt_file = cfg_beavers.path_dataset / 'selection' / 'headwater_reservoirs.txt'
with open(txt_file, 'w', encoding='utf-8') as file:
    for item in headwater_ids:
        file.write(f'{item}\n')

### Plot the graph

In [ ]:
plot_graph(
    graph=G,
    nodes=G.nodes,
    labels=False,
    streams=streams,
    basins=basins,
    title='Of Camels and Beavers',
)